In [34]:
# Create a variable w/ the model S3 URL

#s3://sagemaker-us-east-1-[REDACTED]/testModel-sklearn-byo-model/model.tar.gz
# The name of your S3 bucket:
s3_bucket = "sagemaker-us-east-1-[REDACTED]"
# The directory within your S3 bucket your model is stored in:
bucket_prefix = "testModel-sklearn-byo-model"
# The file name of your model artifact:
model_filename = "model.tar.gz"
# Relative S3 path:
model_s3_key = f"{bucket_prefix}/"+model_filename
# Combine bucket name, model file name, and relate S3 path to create S3 model URL:
model_url = f"s3://{s3_bucket}/{model_s3_key}"

#grab this from artifact upload process, delete above stuff
model_data= "s3://sagemaker-us-east-1-[REDACTED]/testModel-byo-model/test_model.tar.gz"

print(model_data)
# import subprocess

# def authenticate_ecr(region, aws_account_id):
#     cmd = f"aws ecr get-login-password --region {region} | docker login --username AWS --password-stdin {aws_account_id}.dkr.ecr.{region}.amazonaws.com"
#     subprocess.run(cmd, shell=True, check=True)

# # Example usage:
# aws_account_id = "[REDACTED]"
# authenticate_ecr(region, aws_account_id)



s3://sagemaker-us-east-1-[REDACTED]/testModel-byo-model/test_model.tar.gz


In [38]:
ecr_repository = "custom-logistic-model"
region = "us-east-1"
role = "arn:aws:iam::[REDACTED]:role/service-role/AmazonSageMakerServiceCatalogProductsUseRole"
repository_name = ecr_repository

endpoint_name='custom-logistic-model-endpoint'

In [37]:
import boto3

def list_ecr_images(repository_name, region):
    try:
        ecr_client = boto3.client('ecr', region_name=region)
        response = ecr_client.list_images(repositoryName=repository_name)
        image_ids = response.get('imageIds', [])
        if image_ids:
            print(f"Images in repository '{repository_name}':")
            for image_id in image_ids:
                print(f"  - Image digest: {image_id['imageDigest']}")
        else:
            print(f"No images found in repository '{repository_name}'.")
    except Exception as e:
        print(f"Error listing images: {str(e)}")

if __name__ == "__main__":
    list_ecr_images(repository_name, region)


Images in repository 'custom-logistic-model':
  - Image digest: sha256:99f80fdb9c3b1aff3244bd96c509696f3d14177fa7990507a0e80c67ed744ac5


In [27]:
#need to delete old image
#remember to use same image id i guess

# def delete_ecr_image(repository_name, image_digest):
#     try:
#         # Delete the image
#         response = ecr_client.batch_delete_image(
#             repositoryName=repository_name,
#             imageIds=[
#                 {
#                     'imageDigest': image_digest
#                 },
#             ]
#         )
#         print(f"Image with digest {image_digest} deleted successfully.")
#     except Exception as e:
#         print(f"Error deleting image: {str(e)}")

# # Example usage: Delete an image with a specific digest
# image_digest_to_delete = 'sha256:3d00b45df0cfb9d539eea3daf624c8968f4afcdd7556eb3bbf1a7090f65092f5'  # Replace with the actual image digest
# delete_ecr_image(repository_name, image_digest_to_delete)
#doesn't work bc no IAM delete policy attached -- just manually delted for now

In [29]:
#old response
response

{'imageIds': [{'imageDigest': 'sha256:3d00b45df0cfb9d539eea3daf624c8968f4afcdd7556eb3bbf1a7090f65092f5'},
  {'imageDigest': 'sha256:a13c6ce51cde20ed985e48ee86f524141b4eba741a62dbc321314a842540ec53',
   'imageTag': 'latest'}],
 'ResponseMetadata': {'RequestId': '850b4a2c-8826-42b6-958e-e16b24cbc423',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '850b4a2c-8826-42b6-958e-e16b24cbc423',
   'date': 'Tue, 02 Jul 2024 01:20:36 GMT',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '214',
   'connection': 'keep-alive'},
  'RetryAttempts': 0}}

In [30]:
ecr_client = boto3.client('ecr', region_name=region)
response = ecr_client.list_images(repositoryName=repository_name)
ecr_image_tag = response['imageIds'][0]['imageTag']

imageDigest = response['imageIds'][0]['imageDigest']
ecr_image_tag



'latest'

In [39]:
#these are inconsistent -- idk why
# response['imageDetails'][0]['imagePushedAt'], response['imageDetails'][0]['lastRecordedPullTime']

In [32]:
response = ecr_client.describe_images(repositoryName=repository_name)
registryId = response['imageDetails'][0]['registryId']
registryId

'[REDACTED]'

In [33]:
# image_uri = '[REDACTED].dkr.ecr.us-east-1.amazonaws.com/[REDACTED]-ml'

image_uri = registryId + '.dkr.ecr.us-east-1.amazonaws.com/'+ repository_name
image_uri

'[REDACTED].dkr.ecr.us-east-1.amazonaws.com/[REDACTED]-ml'

In [ ]:
# # Import model into hosting
# container = get_image_uri(boto3.Session().region_name, "xgboost", "0.90-2")
# print(container)


In [7]:
# import boto3

# # Initialize the Boto3 client for ECS
# ecs_client = boto3.client('ecs')

# cluster_name = '[REDACTED]ML-ecs-cluster'
# service_name = '[REDACTED]ML-ecs-service'
# task_definition = '[REDACTED]ML-task-definition'
# container_name = '[REDACTED]ML-container'


# ecr_image_uri = f"{ecr_repository}:{ecr_image_tag}"

# response = ecs_client.update_service(
#     cluster=cluster_name,
#     service=service_name,
#     taskDefinition=task_definition,
#     forceNewDeployment=True
# )
# print(f"Service {service_name} updated with the new task definition.")


ClientException: An error occurred (ClientException) when calling the UpdateService operation: TaskDefinition not found.

In [ ]:
# response = ecs_client.run_task(
#     cluster=cluster_name,
#     taskDefinition=task_definition,
#     launchType='EC2',
#     overrides={
#         'containerOverrides': [
#             {
#                 'name': container_name,
#                 'environment': [
#                     {
#                         'name': 'ECR_IMAGE_URI',
#                         'value': ecr_image_uri
#                     }
#                 ]
#             }
#         ]
#     }
# )
# print(f"New task started with the updated task definition using ECR image {ecr_image_uri}.")


In [35]:
from sagemaker.model import Model
model = Model(model_data=model_data, role=role, image_uri=image_uri)

import time
starttime=time.time()

predictor = model.deploy(instance_type='ml.m5.large', initial_instance_count=1, 
                         endpoint_name=endpoint_name)

print(time.time()-starttime, ' secs')

------------------------------------------------*

UnexpectedStatusException: Error hosting endpoint test-endpoint-from-custom-pipeline: Failed. Reason: The primary container for production variant AllTraffic did not pass the ping health check. Please check CloudWatch logs for this endpoint.. Try changing the instance type or reference the troubleshooting page https://docs.aws.amazon.com/sagemaker/latest/dg/async-inference-troubleshooting.html

In [ ]:
print(time.time()-starttime, ' secs')

In [56]:
import boto3
from datetime import datetime
from sagemaker.compute_resource_requirements.resource_requirements import ResourceRequirements
from sagemaker.predictor import Predictor
from sagemaker.enums import EndpointType
from sagemaker.model import Model
from sagemaker.session import Session

resources = ResourceRequirements(
    requests = {
        "num_cpus": 2,  # Number of CPU cores required:
        "num_accelerators": 1, # Number of accelerators required
        "memory": 8192,  # Minimum memory required in Mb (required)
        "copies": 1,
    },
    limits = {},
)

now = datetime.now()
dt_string = now.strftime("%d-%m-%Y-%H-%M-%S")
model_name = "deployModel"+dt_string
#the ECR image URI (e.g., <your-account-id>.dkr.ecr.<your-region>.amazonaws.com/<repository-name>:<tag>)
image_uri = registryId +'.dkr.ecr.'+region+'.amazonaws.com/'+ecr_repository+':'+imageTag



# build your model with Model class
model = Model(
    name = model_name,
    model_data = model_data,
    role = role,
    resources = resources,
    predictor_cls = Predictor,
)
           

import time
starttime=time.time()


predictor = model.deploy(
    initial_instance_count = 1,
    instance_type = "ml.m5.large", 
    endpoint_type = EndpointType.INFERENCE_COMPONENT_BASED,
    resources = resources,
)

#predictor = model.deploy(instance_type="ml.t2.medium", initial_instance_count=1)

print(time.time()-starttime, ' secs')

-----!

TypeError: expected string or bytes-like object

In [14]:
from sagemaker.sklearn import SKLearnModel
sklearn_model = SKLearnModel(model_data=model_data,
                             role=role,
                             entry_point="inference.py",
                             framework_version="1.0-1")

#image uri is docker image file location --> AWS ECR??

import time
starttime=time.time()
predictor = sklearn_model.deploy(instance_type="ml.m5.large", initial_instance_count=1)
print(time.time()-starttime, ' secs.')

FileNotFoundError: [Errno 2] No such file or directory: 'inference.py'

In [ ]:
#TRY RERUNNING WITH TAG
image_uri = '[REDACTED].dkr.ecr.us-east-1.amazonaws.com/[REDACTED]-ml:latest'

from sagemaker.sklearn import SKLearnModel
sklearn_model = SKLearnModel(model_data=model_data,
                             role=role,
                             image_uri=image_uri)

#image uri is docker image file location --> AWS ECR??

import time
starttime=time.time()
predictor = sklearn_model.deploy(instance_type="ml.m5.large", initial_instance_count=1)
print(time.time()-starttime, ' secs.')

--------------------------------------

In [27]:
print(time.time()-starttime, ' secs.')

1577.6263484954834  secs.


In [33]:
import boto3
client = boto3.client('sagemaker')

# List all endpoints
endpoints = client.list_endpoints()

endpoints


{'Endpoints': [{'EndpointName': '[REDACTED]-ml-2024-06-07-05-10-40-087',
   'EndpointArn': 'arn:aws:sagemaker:us-east-1:[REDACTED]:endpoint/[REDACTED]-ml-2024-06-07-05-10-40-087',
   'CreationTime': datetime.datetime(2024, 6, 7, 5, 10, 40, 678000, tzinfo=tzlocal()),
   'LastModifiedTime': datetime.datetime(2024, 6, 7, 5, 34, 24, 261000, tzinfo=tzlocal()),
   'EndpointStatus': 'Failed'},
  {'EndpointName': 'testModel30-05-2024-08-55-37-2024-05-30-08-55-42-199',
   'EndpointArn': 'arn:aws:sagemaker:us-east-1:[REDACTED]:endpoint/testModel30-05-2024-08-55-37-2024-05-30-08-55-42-199',
   'CreationTime': datetime.datetime(2024, 5, 30, 8, 55, 42, 811000, tzinfo=tzlocal()),
   'LastModifiedTime': datetime.datetime(2024, 5, 30, 8, 58, 7, 542000, tzinfo=tzlocal()),
   'EndpointStatus': 'InService'},
  {'EndpointName': 'testModel30-05-2024-08-09-30-2024-05-30-08-09-40-676',
   'EndpointArn': 'arn:aws:sagemaker:us-east-1:[REDACTED]:endpoint/testModel30-05-2024-08-09-30-2024-05-30-08-09-40-676',
  

In [38]:

# Print endpoint names and their statuses
for endpoint in endpoints['Endpoints']:
    print(f"Endpoint Name: {endpoint['EndpointName']}, Status: {endpoint['EndpointStatus']}")


Endpoint Name: [REDACTED]-ml-2024-06-07-05-10-40-087, Status: Failed
Endpoint Name: testModel30-05-2024-08-55-37-2024-05-30-08-55-42-199, Status: InService
Endpoint Name: testModel30-05-2024-08-09-30-2024-05-30-08-09-40-676, Status: InService
Endpoint Name: model-name-2024-05-29-23-18-05-226, Status: InService
Endpoint Name: model-name-2024-05-29-23-09-09-456, Status: InService
Endpoint Name: model-name-2024-05-24-05-26-18-931, Status: InService
Endpoint Name: sagemaker-scikit-learn-2024-05-23-22-03-28-102, Status: Failed


In [40]:
#kill all endpoints
for endpoint in endpoints['Endpoints']:
    if endpoint['EndpointStatus']=='InService':
        print('terminating ',endpoint['EndpointName'])
        response = client.delete_endpoint(EndpointName=endpoint['EndpointName'])


terminating  testModel30-05-2024-08-55-37-2024-05-30-08-55-42-199
terminating  testModel30-05-2024-08-09-30-2024-05-30-08-09-40-676
terminating  model-name-2024-05-29-23-18-05-226
terminating  model-name-2024-05-29-23-09-09-456
terminating  model-name-2024-05-24-05-26-18-931


In [42]:
#relist
endpoints = client.list_endpoints()

for endpoint in endpoints['Endpoints']:
    print(f"Endpoint Name: {endpoint['EndpointName']}, Status: {endpoint['EndpointStatus']}")


Endpoint Name: [REDACTED]-ml-2024-06-07-05-10-40-087, Status: Failed
Endpoint Name: sagemaker-scikit-learn-2024-05-23-22-03-28-102, Status: Failed


In [43]:
#kill all endpoints
for endpoint in endpoints['Endpoints']:
    print('terminating ',endpoint['EndpointName'])
    response = client.delete_endpoint(EndpointName=endpoint['EndpointName'])

    endpoints = client.list_endpoints()

for endpoint in endpoints['Endpoints']:
    print(f"Endpoint Name: {endpoint['EndpointName']}, Status: {endpoint['EndpointStatus']}")


terminating  [REDACTED]-ml-2024-06-07-05-10-40-087
terminating  sagemaker-scikit-learn-2024-05-23-22-03-28-102
Endpoint Name: sagemaker-scikit-learn-2024-05-23-22-03-28-102, Status: Deleting


In [ ]:
#write code here later to test response from cluster

In [29]:
predictor

NameError: name 'predictor' is not defined

In [ ]:
predictor.delete_endpoint()

In [ ]:
#Discard rest below, failed attempts

In [17]:
!pygmentize ./code/inference.py

import os
import joblib

def predict_fn(input_object, model):
    ###########################################
    # Do your custom preprocessing logic here #
    ###########################################

    print("calling model")
    predictions = model.predict(input_object)
    return predictions


def model_fn(model_dir):
    print("loading model.joblib from: {}".format(model_dir))
    loaded_model = joblib.load(os.path.join(model_dir, "model.joblib"))
    return loaded_model


In [18]:
!pygmentize ./code/requirements.txt

boto3
requests
nltk


In [20]:
model = SKLearnModel(
    role=role,
    model_data=model_data,
    framework_version="1.2-1",
    py_version="py3",
    source_dir="code",
    entry_point="inference.py",
)

In [21]:
%%time

predictor = sklearn_model.deploy(instance_type="ml.m5.large", initial_instance_count=1)


FileNotFoundError: [Errno 2] No such file or directory: 'inference.py'

In [ ]:
predictor.delete_endpoint()

In [11]:
import boto3
from sagemaker import Session

# Initialize the SageMaker session
sagemaker_session = Session()

# List all endpoints
endpoints = sagemaker_session.list_endpoints()

# Print endpoint names and their statuses
for endpoint in endpoints:
    print(f"Endpoint Name: {endpoint['EndpointName']}, Status: {endpoint['EndpointStatus']}")


AttributeError: 'Session' object has no attribute 'list_endpoints'